In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("ETL-Streaming-Titles-Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
print("A ler a tabela 'projeto.amazon_titles'...")

df_amazon = spark.read.table("projeto.amazon_titles")

# Adiciona a coluna 'platform' com o valor 'amazon'
df_amazon_platform = df_amazon.withColumn("platform", lit("amazon"))

print(f"Registos lidos da Amazon: {df_amazon_platform.count()}")
df_amazon_platform.select("id", "title", "platform").show(5, truncate=False)

A ler a tabela 'projeto.amazon_titles'...
Registos lidos da Amazon: 9868
+--------+-----------------------+--------+
|id      |title                  |platform|
+--------+-----------------------+--------+
|tm10553 |saleslady              |amazon  |
|tm154438|arrest bulldog drummond|amazon  |
|tm265186|invisible enemy        |amazon  |
|tm5051  |algiers                |amazon  |
|tm5380  |the singing cowgirl    |amazon  |
+--------+-----------------------+--------+
only showing top 5 rows



In [3]:
print("A ler a tabela 'projeto.netflix_titles'...")

df_netflix = spark.read.table("projeto.netflix_titles")

# Adiciona a coluna 'platform' com o valor 'netflix'
df_netflix_platform = df_netflix.withColumn("platform", lit("netflix"))

print(f"Registos lidos da Netflix: {df_netflix_platform.count()}")
df_netflix_platform.select("id", "title", "platform").show(5, truncate=False)

A ler a tabela 'projeto.netflix_titles'...
Registos lidos da Netflix: 5849
+--------+---------------------------------------------------------+--------+
|id      |title                                                    |platform|
+--------+---------------------------------------------------------+--------+
|tm105278|inuyasha the movie 2: the castle beyond the looking glass|netflix |
|tm105329|road to perdition                                        |netflix |
|tm110923|divine intervention                                      |netflix |
|tm111226|awara paagal deewana                                     |netflix |
|tm111600|the sweetest thing                                       |netflix |
+--------+---------------------------------------------------------+--------+
only showing top 5 rows



In [4]:
print("--- Schema Amazon (com plataforma) ---")
df_amazon_platform.printSchema()

print("\n--- Schema Netflix (com plataforma) ---")
df_netflix_platform.printSchema()

--- Schema Amazon (com plataforma) ---
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)
 |-- genre_scifi: integer (nullable = true)
 |-- genre_documentation: integer (nullable = true)
 |-- genre_crime: integer (nullable = true)
 |-- genre_fantasy: integer (nullable = true)
 |-- genre_action: integer (nullable = true)
 |-- genre_animation: integer (nullable = true)
 |-- genre_sport: integer (nullable = true)
 |-- genre_family: integer (nullable = true)
 |-- genre_horror: integer (nullable = true)
 |-- g

In [5]:
df_streaming_titles = df_amazon_platform.unionByName(df_netflix_platform)

print("União concluída.")

União concluída.


In [6]:
count_amazon = df_amazon_platform.count()
count_netflix = df_netflix_platform.count()
count_total = df_streaming_titles.count()

print(f"Registos Amazon:   {count_amazon}")
print(f"Registos Netflix:  {count_netflix}")
print(f"Registos Totais:   {count_total}")

is_match = (count_amazon + count_netflix) == count_total
print(f"\nVerificação (Soma == Total?): {is_match}")

if not is_match:
    print("ALERTA: A soma das contagens não bate com o total!")

Registos Amazon:   9868
Registos Netflix:  5849
Registos Totais:   15717

Verificação (Soma == Total?): True


In [7]:
print("Contagem final por plataforma:")
df_streaming_titles.groupBy("platform").count().show()

print("\nSchema final unificado:")
df_streaming_titles.printSchema()

print("\nAmostra de dados unificados:")
df_streaming_titles.select("id", "title", "platform", "release_year").show(10)

Contagem final por plataforma:
+--------+-----+
|platform|count|
+--------+-----+
|  amazon| 9868|
| netflix| 5849|
+--------+-----+


Schema final unificado:
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)
 |-- genre_scifi: integer (nullable = true)
 |-- genre_documentation: integer (nullable = true)
 |-- genre_crime: integer (nullable = true)
 |-- genre_fantasy: integer (nullable = true)
 |-- genre_action: integer (nullable = true)
 |-- genre_animation: integer (nullable = true)
 |-- genre_sport: in

In [8]:
df_streaming_titles \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/projeto/silver/streaming_titles")

In [9]:
df_streaming_titles.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("release_year") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/streaming_titles") \
    .saveAsTable("projeto.streaming_titles")

In [10]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.streaming_titles
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2025-11-17 21:40:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|       null|  Serializable|        false|{numFiles -> 224,...|        null|Apache-Spark/3.4....|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+-----------

In [11]:
spark.sql(
    """
    SELECT * FROM projeto.streaming_titles
    """
).show()

+--------+--------------------+-----+--------------------+------------+-----------------+-------+-------+---------+----------+----------+---------------+----------+-----------+-------------------+-----------+-------------+------------+---------------+-----------+------------+------------+-------------+-----------+-----------+-------------+---------+--------------+-------------+--------------+-------------+------------+--------------------+--------+
|      id|               title| type|         description|release_year|age_certification|runtime|seasons|  imdb_id|imdb_score|imdb_votes|tmdb_popularity|tmdb_score|genre_scifi|genre_documentation|genre_crime|genre_fantasy|genre_action|genre_animation|genre_sport|genre_family|genre_horror|genre_history|genre_music|genre_drama|genre_western|genre_war|genre_european|genre_romance|genre_thriller|genre_reality|genre_comedy|production_countries|platform|
+--------+--------------------+-----+--------------------+------------+-----------------+-----

In [12]:
spark.stop()